# 검색 성능 최적화를 위한 벡터DB 구축

In [1]:

from dotenv import load_dotenv
import os

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

## Pinecone Index 생성

In [ ]:
from pinecone import Pinecone, ServerlessSpec # Pinecone 클라이언트, 인덱스 스펙설정

pc = Pinecone()
print(pc.list_indexes().names()) # 인덱스 이름

# 인덱스명 ir
if 'ir' not in pc.list_indexes().names():
    # 인덱스 생성
    pc.create_index(
        name = 'ir',
        dimension = 1536,
        metric = 'cosine',       # 유사도 기준
        spec = ServerlessSpec(
            region= 'us-east-1',
            cloud= 'aws'
        )
    )
    print('ir 인덱스 생성 완료!')
else:
    print('ir 인덱스 이미 존재!')

[]
ir 인덱스 생성 완료!


### VectorStore 연결

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model='text-embedding-3-small') # 1536차원 임베딩 모델
# 벡터스토어 연결
vector_store = PineconeVectorStore(
    index_name = 'ir', # 연결할 index명
    embedding = embeddings # 사용할 임베딩 함수 (연결할 인덱스 차원과 임베딩 차원이 같아야함)
)

## Document Upsert

In [4]:
import pandas as pd

df = pd.read_csv('documents.csv')
df

,idx,doc_id,title,content
0,0,D1,제주도 여행 가이드,"제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(..."
1,1,D2,전주 비빔밥과 진주 비빔밥 차이점,"비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기..."
2,2,D3,걸스데이 히트곡 분석,"걸스데이는 2010년 데뷔한 대한민국의 4인조 걸그룹으로, 대표곡으로는 “Somet..."
3,3,D4,세종대왕과 훈민정음,세종대왕(1397~1450)은 훈민정음을 창제하여 한글을 보급한 조선의 4대 임금입...
4,4,D5,이순신 장군의 명량 해전,이순신 장군(1545~1598)은 임진왜란 당시 명량 해전에서 13척의 배로 133...
5,5,D6,2024년 기후 변화 종합 보고서,"2024년 전 지구 평균 기온은 산업화 이전 대비 약 1.2℃ 상승했으며, 해수면 ..."
6,6,D7,AI 기술 동향 및 윤리,"최근 인공지능 분야에서는 생성형 AI, 멀티모달 모델, 강화학습 기반 에이전트 개발..."
7,7,D8,서울 지하철 이용 가이드,"서울 지하철은 1호선부터 9호선까지 운행되며, 주요 환승역으로는 서울역·강남역·종로..."
8,8,D9,판소리 “춘향가” 서사 구조,"판소리는 소리꾼과 고수가 함께 공연하는 한국 전통 음악으로, 대표 작품에는 “춘향가..."
9,9,D10,한국 축구 대표팀 주요 기록,"한국 축구 대표팀은 2002 한일 월드컵 4강 진출, 2012 런던 올림픽 동메달 ..."


In [ ]:
# DataFrame -> Langchain Document 변환
from langchain_core.documents import Document # Langchain 문서 객체

docs_to_index = []

for idx, row in df.iterrows(): # df를 행 단위로 순회
    doc_id = row['doc_id']     # 문서 ID 추출
    content = row['content']   # 문서 본문 추출
    doc = Document(page_content=content, metadata={'doc_id': doc_id}) # Document 형식 저장
    docs_to_index.append(doc)

print(len(docs_to_index))

30


In [7]:
# 벡터스토어에 문서 업서트 : 임베딩 후 저장(기존에 있으면 Update, 없으면 Insert)
vector_store.add_documents(docs_to_index)

['88a51ed0-c8b1-484a-bf27-671fd4248b57',
 '4a76cacf-ef2a-4c6a-a5e2-edb741a91fc4',
 'fd55ec50-2d61-444a-a8e6-50ce7165e961',
 '8a93a02f-8256-4ac1-95bc-58b41b9f0520',
 '8586852c-fd95-4aa6-a667-9cce31784afb',
 '8b69698c-b1fd-408d-880d-b214349b0e38',
 '0601d0a2-885a-4cee-845d-723977f4e51b',
 '895b1410-62d6-491e-9876-3543d06a0317',
 'e10b599d-fa74-4adb-b5b2-e73047122ebf',
 'c3bc1607-4794-4495-b659-82eb6aa80138',
 'caffa651-45e3-446c-9d04-594d9553e562',
 'f41448f8-ee78-4123-9664-cd5a1edb7ebc',
 '3fa8c2d0-6375-43ba-85f8-67329361f93a',
 'd254f6f2-4f5d-47d0-97a2-f1dcba415262',
 'b0742e3f-d6de-414e-ae27-9881ea28abc6',
 'd5d8d336-1a94-4746-aaf1-ec3e2e995c46',
 '0e6916b7-9d2a-4a46-ab36-5c2d4e923d9e',
 'ea1f7fb5-4a89-4e10-aa80-9d0301de442c',
 '712a9b6d-ad72-4ba3-93ea-7cfaa331c689',
 'fd2b50d0-f151-41ab-bf10-8f627098030d',
 'eeb4a8fa-4ff7-417a-9669-c266ad0a2420',
 'b600e7f4-4a0b-4448-9029-b6d36894ecfe',
 '71562934-446d-43e1-a671-4dba2619881c',
 '55e94b6e-b6c2-488d-8995-d4f5a5f58c3d',
 '6ac8636d-b9aa-

In [ ]:
query = '제주도 여행'

results = vector_store.similarity_search_with_score(query, k=5) # [(유사 문서1, 점수), ...]
for rank, (doc, score) in enumerate(results, 1):
    print(f"{rank} Score: {score}")
    print(f"doc_id: {doc.metadata['doc_id']}")
    print(f"content: {doc.page_content}")
    

1 Score: 0.508254111
doc_id: D1
content: 제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(...
2 Score: 0.439489365
doc_id: D12
content: 서울 근교에서 당일치기로 다녀올 만한 여행지로는 가평 쁘띠프랑스, 남양주 수종사, ...
3 Score: 0.281069219
doc_id: D13
content: 비빔밥은 지역별로 칼로리, 탄수화물, 단백질, 지방 함량이 차이를 보입니다. 전주 ...
4 Score: 0.23558338
doc_id: D17
content: 2023년 한국 영화 흥행 순위 Top10에는 “헌트”, “비상선언”, “범죄도시3...
5 Score: 0.20054765
doc_id: D9
content: 판소리는 소리꾼과 고수가 함께 공연하는 한국 전통 음악으로, 대표 작품에는 “춘향가...
